# Step 6 — manual stop additions

Stations added by hand on top of step 5's night train stops, so the catalog
covers places no night train serves *today* but a target network would: large
urban areas without a qualified stop, tourism regions, and major ferry hubs.

**This notebook is the source of truth for those additions.** The selections
live in the `ADDITIONS` cells below, grouped by region and keyed by OSM stop
id, and everything else (name, coordinates, country) is looked up from step 3b
and step 4 at run time — so a stop is added or removed by editing one line
here, and `git diff` shows exactly what changed and why.

## The `reason` field

Every addition carries a `reason` of the form `criterion` or `criterion:place`.
The design requires that *"why is station X (not) included?"* be answerable
from the data alone, including by people outside the project. The vocabulary:

| reason | when |
|---|---|
| `fua:<city>` | functional urban area with no qualified stop — name the city |
| `tourism:<region>` | tourism destination — name the region |
| `ferry:<port>` | major ferry hub — name the port |
| `border` | border/interchange station |
| `network` | needed to make a corridor coherent |
| `night_train_stop` | served today but missing from ONTD/step 5 |

`<city>` is a placeholder to **replace**, not to keep: `fua:Bayreuth`, not
`fua:<city>`. The report at the bottom counts placeholders as unfilled.

## Guards

The resolve cell refuses to write the output while any addition

- points at an OSM id step 3b does not know,
- points at a metro/tram/bus/funicular object instead of the railway station
  (`station_mode` check; ferry piers pass only with a `ferry:` reason), or
- duplicates a step 5 stop — same OSM id, same ONTD station through a
  different OSM object, or within 300 m of a qualified stop.

Those three are how the catalog once ended up with two Gesundbrunnens; they
are errors, not judgement calls. What *is* a judgement call — an `fua:` reason
where the same urban area meanwhile has a qualified stop — is written to
`data/step6_overlap_review.csv` for review instead of blocking the run.

## Output

`data/step6_manual_additions.csv` — `stop_id, stop_name, country, stop_lat,
stop_lon, reason` — consumed by `step10_export_seed_stops.py`, which unions it
with the current step 5 output.


In [1]:
import csv
import math
from collections import Counter

from data_sources import DATA_DIR, ensure_local, local_input

OUTPUT_PATH = DATA_DIR / "step6_manual_additions.csv"
OVERLAP_REVIEW_PATH = DATA_DIR / "step6_overlap_review.csv"


## The additions

One dict per region, `stop_id: (name, reason)`. The name is a comment for
readability only — it is re-read from step 3b when the file is written, so a
stale name here cannot corrupt the output. Add a stop by adding a line; remove
one by deleting its line.

In [2]:
# Germany, Austria, Switzerland
ADDITIONS_GERMANY = {
    # --- DE ---
    "osm:n31485922": ("Bayreuth Hbf", "fua:<city>"),
    "osm:n1874501382": ("Bielefeld Hauptbahnhof", "fua:<city>"),
    "osm:w24806780": ("Braunschweig Hauptbahnhof", "fua:<city>"),
    "osm:n26562398": ("Bremerhaven Hauptbahnhof", "fua:<city>"),
    "osm:n2711388096": ("Böblingen", "fua:<city>"),
    "osm:n3607858763": ("Chemnitz Hauptbahnhof", "fua:<city>"),
    "osm:n2599505466": (
        "Cottbus Hauptbahnhof / Chóśebuz głowne dwórnišćo",
        "fua:<city>",
    ),
    "osm:n3175310444": ("Darmstadt Hauptbahnhof", "fua:<city>"),
    "osm:n4189000814": ("Flensburg / Flensborg", "fua:<city>"),
    "osm:n1840958277": ("Gera Hauptbahnhof", "fua:<city>"),
    "osm:n4349314485": ("Göppingen", "fua:<city>"),
    "osm:n1438696887": ("Görlitz", "fua:<city>"),
    "osm:n3450444902": ("Gütersloh Hbf", "fua:<city>"),
    "osm:n2820941790": ("Heidelberg Hauptbahnhof", "fua:<city>"),
    "osm:n27385328": ("Heilbronn Hauptbahnhof", "fua:<city>"),
    "osm:n3616040153": ("Hildesheim Hauptbahnhof", "fua:<city>"),
    "osm:n3543804400": ("Ingolstadt Hbf", "fua:<city>"),
    "osm:n1126168394": ("Jena Paradies", "fua:<city>"),
    "osm:n30959690": ("Kaiserslautern Hauptbahnhof", "fua:<city>"),
    "osm:n4530820004": ("Kempten (Allgäu) Hbf", "fua:<city>"),
    "osm:n4257641280": ("Kiel Hauptbahnhof", "fua:<city>"),
    "osm:n534753716": ("Landshut (Bay) Hbf", "fua:<city>"),
    "osm:n3087634633": ("Magdeburg Hauptbahnhof", "fua:<city>"),
    "osm:n7160009313": ("Neumünster", "fua:<city>"),
    "osm:n91753264": ("Oldenburg (Oldb) Hbf", "fua:<city>"),
    "osm:n268894281": ("Osnabrück Hauptbahnhof", "fua:<city>"),
    "osm:n2675283037": ("Paderborn Hauptbahnhof", "fua:<city>"),
    "osm:n25972727": ("Pforzheim Hauptbahnhof", "fua:<city>"),
    "osm:n1755712810": ("Plauen (Vogtl) ob Bf", "fua:<city>"),
    "osm:n25233549": ("Reutlingen Hbf", "fua:<city>"),
    "osm:n987773654": ("Rostock Hauptbahnhof", "fua:<city>"),
    "osm:n259449966": ("Saarbrücken Hauptbahnhof", "fua:<city>"),
    "osm:n27381920": ("Schweinfurt Hbf", "fua:<city>"),
    "osm:n252098248": ("Schwerin Hauptbahnhof", "fua:<city>"),
    "osm:n277350630": ("Stralsund Hbf", "tourism:<region>"),
    "osm:n745097775": ("Trier Hbf", "fua:<city>"),
    "osm:n338899629": ("Wolfsburg Hauptbahnhof", "fua:<city>"),
    # --- CH ---
    "osm:n2051794005": ("Luzern", "fua:<city>"),
    "osm:n2428167137": ("Schaffhausen", "fua:<city>"),
    "osm:n1346888802": ("St. Gallen", "fua:<city>"),
    "osm:n3081154442": ("Thun", "fua:<city>"),
    "osm:n1286602751": ("Winterthur", "fua:<city>"),
    "osm:n60093107": (
        "Wien Westbahnhof",
        "network — major Vienna terminus, referenced by route fixtures",
    ),
}

In [3]:
# France, Benelux
ADDITIONS_FRANCE = {
    "osm:n269296749": ("Marne-la-Vallée Chessy", "tourism:Disneyland Paris"),
    # --- FR ---
    "osm:n4290854846": ("Aix-en-Provence", "fua:<city>"),
    "osm:n1680885216": ("Amiens", "fua:<city>"),
    "osm:n3486353293": ("Angers Saint-Laud", "fua:<city>"),
    "osm:n5061961433": ("Annecy", "fua:<city>"),
    "osm:n7167504997": ("Arras", "fua:<city>"),
    "osm:n3805976209": ("Avignon-Centre", "fua:<city>"),
    "osm:n2501252269": ("Belfort", "fua:<city>"),
    "osm:n2500070617": ("Besançon-Viotte", "fua:<city>"),
    "osm:n8213648712": ("Boulogne Ville", "fua:<city>"),
    "osm:n194212267": ("Bourges", "fua:<city>"),
    "osm:n2207570062": ("Brest", "tourism:<region>"),
    "osm:w73575850": ("Briançon", "fua:<city>"),
    "osm:n5598384401": ("Caen", "fua:<city>"),
    "osm:n9183304884": ("Calais-Ville", ""),
    "osm:n5066478129": ("Chambéry - Challes-les-Eaux", "fua:<city>"),
    "osm:n8303862255": ("Chartres", "fua:<city>"),
    "osm:w112095036": ("Cherbourg", "fua:<city>"),
    "osm:n10936459654": ("Clermont-Ferrand", "fua:<city>"),
    "osm:n398836628": ("Colmar", "fua:<city>"),
    "osm:n11556100824": ("Douai", "fua:<city>"),
    "osm:n312805118": ("Dunkerque", "fua:<city>"),
    "osm:n5070332503": ("Grenoble", "fua:<city>"),
    "osm:n4975049664": ("La Rochelle", "fua:<city>"),
    "osm:n847701543": ("Le Havre", "fua:<city>"),
    "osm:n8745537419": ("Lille-Flandres", "fua:<city>"),
    "osm:n3471044194": ("Limoges-Bénédictins", "fua:<city>"),
    "osm:n10940619366": ("Lorient", "fua:<city>"),
    "osm:n4290857018": ("Lyon Perrache", "fua:<city>"),
    "osm:n11225690169": ("Martigues", "fua:<city>"),
    "osm:n4290857026": ("Metz", "fua:<city>"),
    "osm:n4225150278": ("Montbéliard", "fua:<city>"),
    "osm:n2502268309": ("Mulhouse-Ville", "fua:<city>"),
    "osm:n4290857032": ("Nancy", "fua:<city>"),
    "osm:n3486337796": ("Nantes", "fua:<city>"),
    "osm:n9912502487": ("Poitiers", "fua:<city>"),
    "osm:n4986753876": ("Quimper", "fua:<city>"),
    "osm:n2517400258": ("Reims", "fua:<city>"),
    "osm:n4250849558": ("Rennes", "fua:<city>"),
    "osm:n2483753165": ("Roanne", "fua:<city>"),
    "osm:n2076751841": ("Rouen Rive-Droite", "fua:<city>"),
    "osm:n4280767167": ("Saint-Brieuc", "fua:<city>"),
    "osm:n829527258": ("Saint-Raphaël-Valescure", "fua:<city>"),
    "osm:n2010251922": ("Saint-Étienne Châteaucreux", "fua:<city>"),
    "osm:n3069229440": ("Strasbourg", "fua:<city>"),
    "osm:n2506173917": ("Troyes", "fua:<city>"),
    "osm:n10935384072": ("Valence-Ville", "fua:<city>"),
    "osm:n1648968005": ("Valenciennes", "fua:<city>"),
    "osm:n394710073": ("Vannes", "fua:<city>"),
    # --- BE ---
    "osm:n2929614444": ("Brugge", "fua:<city>"),
    "osm:n1178257779": ("Gent-Sint-Pieters", "fua:<city>"),
    "osm:n21309047": ("Kortrijk", "fua:<city>"),
    "osm:n446059037": ("La Louvière-Centre", "fua:<city>"),
    "osm:n7261826908": ("Mechelen-Nekkerspoel", "fua:<city>"),
    "osm:n1027979508": ("Oostende", "fua:<city>"),
    "osm:n26446051": ("Verviers-Central", "fua:<city>"),
    # --- NL ---
    "osm:n4085675596": ("'s-Hertogenbosch", "fua:<city>"),
    "osm:n4085675598": ("Alkmaar", "fua:<city>"),
    "osm:n4530820010": ("Almelo", "fua:<city>"),
    "osm:n4555468696": ("Almere Centrum", "fua:<city>"),
    "osm:n44727803": ("Arnhem Centraal", "fua:<city>"),
    "osm:n7606786768": ("Assen", "fua:<city>"),
    "osm:n43174364": ("Breda", "fua:<city>"),
    "osm:n4425618606": ("Dordrecht", "fua:<city>"),
    "osm:n4487559980": ("Enschede", "fua:<city>"),
    "osm:n1112410297": ("Groningen", "fua:<city>"),
    "osm:n5252716645": ("Haarlem", "fua:<city>"),
    "osm:n45931397": ("Hengelo", "fua:<city>"),
    "osm:n48162487": ("Leeuwarden", "fua:<city>"),
    "osm:n9604567339": ("Leiden Centraal", "fua:<city>"),
    "osm:n46792197": ("Lelystad Centrum", "fua:<city>"),
    "osm:n5311118145": ("Maastricht", "fua:<city>"),
    "osm:n44061300": ("Nijmegen", "fua:<city>"),
    "osm:n42966392": ("Roosendaal", "fua:<city>"),
    "osm:n4041466061": ("Tilburg", "fua:<city>"),
    "osm:n4555433902": ("Venlo", "fua:<city>"),
    "osm:n4487554970": ("Zwolle", "fua:<city>"),
}

In [4]:
# Iberia
ADDITIONS_IBERIA = {
    # --- ES ---
    "osm:n13782316672": ("A Coruña", "fua:<city>"),
    "osm:w28776478": ("Abando Indalecio Prieto", "fua:<city>"),
    "osm:n10914769161": ("Alacant Terminal", "fua:<city>"),
    "osm:n11016395831": ("Albacete Los Llanos", "fua:<city>"),
    "osm:n2182333421": ("Algeciras-Paco de Lucía", "fua:<city>"),
    "osm:n30546837": ("Avilés", "fua:<city>"),
    "osm:n2962633346": ("Badajoz", "fua:<city>"),
    "osm:n617134268": ("Cartagena", "fua:<city>"),
    "osm:n13894649638": ("Castelló", "fua:<city>"),
    "osm:n13717016710": ("Ciudad Real", "fua:<city>"),
    "osm:n13714976346": ("Cáceres", "fua:<city>"),
    "osm:n259625422": ("Cádiz", "fua:<city>"),
    "osm:n7567516121": ("Córdoba Julio Anguita", "fua:<city>"),
    "osm:n1738646773": ("El Puerto de Santa María", "fua:<city>"),
    "osm:w24930154": ("Ferrol", "fua:<city>"),
    "osm:n7201979115": ("Granada", "fua:<city>"),
    "osm:n6313228293": ("Guadalajara", "fua:<city>"),
    "osm:n5580567331": ("Huelva", "fua:<city>"),
    "osm:n8051540821": ("Huesca", "fua:<city>"),
    "osm:n7747872372": ("Huércal-Viator", "fua:<city>"),
    "osm:n5299078295": ("León", "fua:<city>"),
    "osm:n2459459539": ("Logroño", "fua:<city>"),
    "osm:w86123754": ("Lugo", "fua:<city>"),
    "osm:n13017334754": ("Murcia del Carmen", "fua:<city>"),
    "osm:n2609534280": ("Málaga María Zambrano", "fua:<city>"),
    "osm:n2039781019": ("Mérida", "fua:<city>"),
    "osm:n1842017741": ("Ourense-Empalme", "fua:<city>"),
    "osm:n4586092220": ("Oviedo / Uviéu", "fua:<city>"),
    "osm:n1939943099": ("Palencia", "fua:<city>"),
    "osm:n11757382798": ("Pamplona / Iruña", "fua:<city>"),
    "osm:n1069592576": ("Ponferrada", "fua:<city>"),
    "osm:n453651815": ("Pontevedra", "fua:<city>"),
    "osm:n4448930346": ("Salamanca", "fua:<city>"),
    "osm:n2340360836": ("San Fernando-Bahía Sur", "fua:<city>"),
    "osm:n5893228216": ("Santander", "fua:<city>"),
    "osm:n12768629140": ("Santiago de Compostela - Daniel Castelao", "fua:<city>"),
    "osm:n191262271": ("Sevilla - Santa Justa", "fua:<city>"),
    "osm:n2328131203": ("Talavera de la Reina", "fua:<city>"),
    "osm:n13894696022": ("Tarragona", "fua:<city>"),
    "osm:n7246471728": ("València - Estació del Nord", "fua:<city>"),
    "osm:n7246471727": ("València Joaquín Sorolla", "fua:<city>"),
    "osm:n12819737430": ("Vigo-Urzáiz", "fua:<city>"),
    "osm:n29568804": ("Vitoria-Gasteiz", "fua:<city>"),
    "osm:n791063229": ("Zamora", "fua:<city>"),
    # --- PT ---
    "osm:n10783341030": ("Aveiro", "fua:<city>"),
    "osm:n1393070418": ("Barroselas", "fua:<city>"),
    "osm:n10783341036": ("Braga", "fua:<city>"),
    "osm:n10783341029": ("Coimbra-B", "fua:<city>"),
    "osm:n46756926": ("Faro", "fua:<city>"),
    "osm:n10783341033": ("Gaia", "fua:<city>"),
    "osm:n3941781457": ("Guimarães", "fua:<city>"),
    "osm:n6297158592": ("Lisboa - Oriente", "fua:<city>"),
    "osm:n10783341023": ("Porto - Campanhã", "fua:<city>"),
    "osm:n2861321998": ("Viana do Castelo", "fua:<city>"),
}

In [5]:
# Italy
ADDITIONS_ITALY = {
    # ONTD has no Roma Ostiense (the station is an OSM relation, invisible to
    # the node-only ONTD export); without this carry, step 5 used to fall back
    # to the Piramide metro object 550 m away.
    "osm:r1821284": ("Roma Ostiense", "night_train_stop"),
    # --- IT ---
    "osm:r1821284": ("Roma Ostiense", "night_train_stop"),  # ONTD has only the adjacent Metro B object; station is a relation
    "osm:n5324492776": ("Acireale", "fua:<city>"),
    "osm:n12291596709": ("Alessandria", "fua:<city>"),
    "osm:n7460292092": ("Ancona", "fua:<city>"),
    "osm:n593748428": ("Arezzo Pescaiola", "fua:<city>"),
    "osm:n8820637017": ("Avellino", "fua:<city>"),
    "osm:n1699232800": ("Barletta", "fua:<city>"),
    "osm:n8607336257": ("Bergamo", "fua:<city>"),
    "osm:n7473059189": ("Campobasso", "fua:<city>"),
    "osm:n1215079672": ("Caserta", "fua:<city>"),
    "osm:n603353943": ("Cefalù", "tourism:<region>"),
    "osm:n258613100": ("Cerignola Campagna", "fua:<city>"),
    "osm:n1279764780": ("Cosenza Vaglio Lise", "fua:<city>"),
    "osm:n2121919441": ("Ferrara", "fua:<city>"),
    "osm:n738159083": ("L'Aquila", "fua:<city>"),
    "osm:n7042610811": ("Milazzo", "fua:<city>"),
    "osm:n726611782": ("Modena", "fua:<city>"),
    "osm:n11802851419": ("Novara", "fua:<city>"),
    "osm:n5836604869": ("Parma", "fua:<city>"),
    "osm:n211030020": ("Pavia", "fua:<city>"),
    "osm:n249236145": ("Perugia", "fua:<city>"),
    "osm:n1862274592": ("Pesaro", "fua:<city>"),
    "osm:n267591085": ("Pescara Centrale", "fua:<city>"),
    "osm:n13742535643": ("Piacenza", "fua:<city>"),
    "osm:n1670700542": ("Pordenone", "fua:<city>"),
    "osm:n842367835": ("Potenza Centrale", "fua:<city>"),
    "osm:n9067692438": ("Ravenna", "fua:<city>"),
    "osm:n82549162": ("Reggio Emilia", "fua:<city>"),
    "osm:n1274172585": ("Sant'Agata di Militello", "tourism:<region>"),
    "osm:n3395226356": ("Torino Porta Nuova", "fua:<city>"),
    "osm:n1764381735": ("Trento", "fua:<city>"),
}

In [6]:
# United Kingdom, Ireland
ADDITIONS_UNITED_KINGDOM = {
    # --- GB ---
    "osm:n7998566986": ("Ashford International", "fua:<city>"),
    "osm:n4461326005": ("Bangor", "fua:<city>"),
    "osm:n12248421687": ("Belfast Grand Central", "fua:<city>"),
    "osm:n6765532062": ("Birmingham New Street", "fua:<city>"),
    "osm:n6634567434": ("Bournemouth", "fua:<city>"),
    "osm:n7209380367": ("Bradford Interchange", "fua:<city>"),
    "osm:n20947173": ("Brighton", "fua:<city>"),
    "osm:n7167271113": ("Bristol Temple Meads", "fua:<city>"),
    "osm:n573566827": ("Cambridge", "fua:<city>"),
    "osm:n3453612249": ("Canterbury West", "fua:<city>"),
    "osm:n6605149666": ("Cardiff Central", "fua:<city>"),
    "osm:n5028607042": ("Chester", "fua:<city>"),
    "osm:n6688385690": ("Dover Priory", "ferry:<port>"),
    "osm:n6013523209": ("Exeter St Davids", "fua:<city>"),
    "osm:n6646199707": ("Gloucester", "fua:<city>"),
    "osm:n7154209250": ("Holyhead", "ferry:<port>"),
    "osm:n6012826246": ("Hull Paragon Interchange", "ferry:<port>"),
    "osm:n7156706693": ("Leeds", "fua:<city>"),
    "osm:n4292139459": ("Leicester", "fua:<city>"),
    "osm:n6960405293": ("Liverpool Lime Street", "fua:<city>"),
    "osm:n5064005964": ("Manchester Piccadilly", "fua:<city>"),
    "osm:n7159380475": ("Milton Keynes Central", "fua:<city>"),
    "osm:n195885858": ("Newcastle", "fua:<city>"),
    "osm:n7158616254": ("Norwich", "fua:<city>"),
    "osm:n324650068": ("Nottingham", "fua:<city>"),
    "osm:n6481707942": ("Oxford", "fua:<city>"),
    "osm:n2612643529": ("Peterborough", "fua:<city>"),
    "osm:n6010790017": ("Portsmouth and Southsea", "ferry:<port>"),
    "osm:n5784212748": ("Sheffield", "fua:<city>"),
    "osm:n638908005": ("Southampton Central", "fua:<city>"),
    "osm:n7140234411": ("Southend Victoria", "fua:<city>"),
    "osm:n6900337987": ("Swansea", "fua:<city>"),
    "osm:n104734": ("Swindon", "fua:<city>"),
    "osm:n6634567442": ("Wolverhampton", "fua:<city>"),
    "osm:n7989407332": ("Wrexham General", "fua:<city>"),
    # --- IE ---
    "osm:n5355226792": ("Cork Kent", "fua:<city>"),
    "osm:n7198337980": ("Dublin Connolly", "fua:<city>"),
    "osm:n6854406241": ("Dublin Heuston", "fua:<city>"),
    "osm:n6854415949": ("Galway Ceannt", "fua:<city>"),
    "osm:n6740364330": ("Limerick Colbert", "fua:<city>"),
    "osm:n6852246817": ("Waterford Plunkett", "fua:<city>"),
}

In [7]:
# Nordics
ADDITIONS_NORDICS = {
    # --- SE ---
    "osm:n7135739559": ("Borås C", "fua:<city>"),
    "osm:r10274650": ("Härnösand resecentrum", "fua:<city>"),
    "osm:w419690632": ("Hässleholm C", "night_train_stop"),
    "osm:w155603926": ("Station Åre", "night_train_stop"),
    # --- DK ---
    "osm:n3419486092": ("Aalborg", "fua:<city>"),
    "osm:n5026479524": ("Aarhus H", "fua:<city>"),
    "osm:n1655765253": ("Hirtshals", "ferry:<port>"),
    # --- FI ---
    "osm:n1716259527": ("Espoo", "fua:<city>"),
    "osm:n259004650": ("Jyväskylä", "fua:<city>"),
    "osm:n603918145": ("Kotka satama", "network"),
    "osm:n292809487": ("Kuopio", "fua:<city>"),
    "osm:n537913195": ("Lahti", "fua:<city>"),
    "osm:n340019021": ("Tikkurila", "fua:<city>"),
    "osm:n91925127": ("Vaasa", "network"),
    "osm:n4993961319": (
        "Esbjerg",
        "ferry:Esbjerg — carried over from the curated catalog, which step 7 now replaces",
    ),
}

In [8]:
# Central Europe
ADDITIONS_CENTRAL_EUROPE = {
    # --- PL ---
    "osm:n3437310369": ("Bydgoszcz Główna", "fua:<city>"),
    "osm:n413346673": ("Elbląg", "fua:<city>"),
    "osm:n2050000245": ("Gorzów Wielkopolski", "fua:<city>"),
    "osm:n3831584572": ("Grudziądz", "fua:<city>"),
    "osm:n3357286930": ("Głogów", "fua:<city>"),
    "osm:n811380395": ("Inowrocław", "fua:<city>"),
    "osm:n3459064660": ("Kalisz", "fua:<city>"),
    "osm:n29830753": ("Konin", "fua:<city>"),
    "osm:n2627870779": ("Legnica", "fua:<city>"),
    "osm:n5356872372": ("Lubin", "fua:<city>"),
    "osm:n475592274": ("Nowy Sącz", "fua:<city>"),
    "osm:n131851982": ("Olsztyn Główny", "fua:<city>"),
    "osm:n528526130": ("Ostrów Wielkopolski", "fua:<city>"),
    "osm:n1864585459": ("Piotrków Trybunalski", "fua:<city>"),
    "osm:n3459469757": ("Piła Główna", "fua:<city>"),
    "osm:n1991734621": ("Płock", "fua:<city>"),
    "osm:n842079165": ("Wałbrzych Miasto", "fua:<city>"),
    "osm:n3459316528": ("Włocławek", "fua:<city>"),
    "osm:n372254443": ("Zamość", "fua:<city>"),
    "osm:n1992495561": ("Łomża", "fua:<city>"),
    # --- CZ ---
    "osm:n3279883031": ("Hradec Králové hlavní nádraží", "fua:<city>"),
    "osm:n5648124921": ("Most", "fua:<city>"),
    "osm:n30077682": ("Plzeň hlavní nádraží", "fua:<city>"),
    # --- SK ---
    "osm:n8012637130": ("Banská Bystrica", "fua:<city>"),
    "osm:n7066101885": ("Nitra", "fua:<city>"),
    "osm:n346420319": ("Prešov", "fua:<city>"),
    "osm:n1911997558": ("Trenčín", "fua:<city>"),
    "osm:n10607895201": ("Zvolen nákladná stanica", "network"),
    "osm:n6446509215": ("Žilina", "fua:<city>"),
    # --- HU ---
    "osm:n268213797": ("Miskolc-Tiszai", "fua:<city>"),
    "osm:n25546152": ("Pécs", "fua:<city>"),
    "osm:n93800956": ("Szombathely", "fua:<city>"),
    "osm:n3129289404": (
        "Česká Třebová",
        "network — junction on the Praha–Brno/Vienna corridor",
    ),
    "osm:n24684084": ("Kolín", "network — junction on the Praha–Brno/Vienna corridor"),
}

In [9]:
# Baltics
ADDITIONS_BALTICS = {
    # --- EE ---
    "osm:n529932898": ("Narva", "fua:<city>"),
    "osm:n30402685": ("Paldiski", "fua:<city>"),
    "osm:n8761131036": ("Tartu", "fua:<city>"),
    # --- LV ---
    "osm:n7800844381": ("Daugavpils", "fua:<city>"),
    "osm:n7799314251": ("Liepāja", "fua:<city>"),
    "osm:n252636397": ("Ventspils-1", "ferry:<port>"),
    # --- LT ---
    "osm:n99172829": ("Klaipėda", "fua:<city>"),
    "osm:n6624987344": ("Vilnius", "fua:<city>"),
    "osm:n1370770391": ("Šiauliai", "fua:<city>"),
}

In [10]:
# South-eastern Europe
ADDITIONS_SOUTH_EASTERN_EUROPE = {
    # --- SI ---
    "osm:n283207146": ("Bled Jezero", "fua:<city>"),
    "osm:n270129111": ("Postojna", "tourism:<region>"),
    # --- HR ---
    "osm:n5668860678": ("Pula", "tourism:<region>"),
    # --- BA ---
    "osm:n2038791178": ("Doboj", "network"),
    "osm:n312280159": ("Mostar", "fua:<city>"),
    "osm:n942406094": ("Sarajevo", "fua:<city>"),
    "osm:n10212563520": ("Zenica", "fua:<city>"),
    # --- RS ---
    "osm:n1601596124": ("Ниш", "fua:<city>"),
    "osm:n893439322": ("Суботица", "fua:<city>"),
    # --- ME ---
    "osm:n10728069934": ("Nikšić", "fua:<city>"),
    # --- MK ---
    "osm:n3407148607": ("Битола", "network"),
    "osm:n1635043504": ("Велес", "fua:<city>"),
    "osm:n134701987": ("Гевгелија", "border"),
    "osm:n408145528": ("Куманово", "fua:<city>"),
    "osm:n9947846021": ("Скопје", "fua:<city>"),
    # --- AL ---
    "osm:n13895194677": ("Durrës", "fua:<city>"),
    "osm:n13895194676": ("Terminali i Transportit Publik Tiranë", "fua:<city>"),
    # --- XK ---
    "osm:n2107256271": ("Ferizaj", "fua:<city>"),
    "osm:n1613271652": ("Prishtinë", "fua:<city>"),
    # --- BG ---
    "osm:w529540422": ("Димитровград", "night_train_stop"),
    "osm:w421067799": ("Силистра", "night_train_stop"),
    "osm:n14055863804": ("Сливен", "night_train_stop"),
    # --- RO ---
    "osm:n1243304538": ("Botoșani", "fua:<city>"),
    "osm:n9244693114": ("Brăila", "fua:<city>"),
    "osm:n487098710": ("Călărași Sud", "fua:<city>"),
    "osm:w272187224": ("Dej Triaj", "night_train_stop"),
    "osm:n345539748": ("Focșani", "fua:<city>"),
    "osm:n8176378047": ("Galați", "fua:<city>"),
    "osm:n247416268": (
        "Iași",
        "night_train_stop (step 6 had Nicolina, a secondary station)",
    ),
    "osm:n303278705": ("Piatra-Neamț", "fua:<city>"),
    "osm:n8607130227": ("Pitești", "fua:<city>"),
    "osm:n516125655": ("Râmnicu Vâlcea", "fua:<city>"),
    "osm:n202022357": ("Sighișoara", "tourism:Sighisoara"),
    "osm:n536752658": ("Slatina", "fua:<city>"),
    "osm:n7905078406": ("Tulcea Oraș", "fua:<city>"),
    "osm:n9554314257": ("Târgoviște", "fua:<city>"),
    "osm:n13577341022": ("Târgu Jiu", "fua:<city>"),
    "osm:n258668249": ("Târgu Mureș", "fua:<city>"),
    # --- GR ---
    "osm:n9643537166": ("Βόλος", "fua:<city>"),
    "osm:n4925130296": ("Θεσσαλονίκη", "fua:<city>"),
    "osm:n6175162599": ("Κατερίνη", "fua:<city>"),
    "osm:n287554537": ("Λάρισα", "fua:<city>"),
    "osm:n308841776": ("Ξάνθη", "fua:<city>"),
    "osm:n9721698903": ("Ρίο", "fua:<city>"),
    "osm:n4883927548": ("Σέρραι", "fua:<city>"),
    "osm:w609979867": ("Αθήνα", "fua:<city>"),
    "osm:n13254223965": (
        "Централна гара Бургас",
        "night_train_stop (absent from ONTD)",
    ),
    "osm:w216482601": (
        "Велико Търново",
        "night_train_stop (Горна Оряховица is the regional hub, but the schedule calls at Veliko Tarnovo)",
    ),
}

In [11]:
# Eastern Europe, Türkiye
ADDITIONS_EASTERN_EUROPE = {
    # --- UA ---
    "osm:n9708140929": ("Алчевськ", "fua:<city>"),
    "osm:n3815614789": ("Бердянськ (експ.)", "fua:<city>"),
    "osm:n9695166282": ("Дебальцеве", "fua:<city>"),
    "osm:n278290881": ("Донецьк", "fua:<city>"),
    "osm:n676673228": ("Евпатория-Курорт", "fua:<city>"),
    "osm:n8594603350": ("Житомир", "fua:<city>"),
    "osm:n2469903575": ("Керчь-Порт", "fua:<city>"),
    "osm:n10175634575": ("Луганськ", "fua:<city>"),
    "osm:n1201406760": ("Маріуполь", "fua:<city>"),
    "osm:n652003827": ("Мелітополь", "fua:<city>"),
    "osm:n2635702849": (
        "Миколаїв",
        "night_train_stop (step 6 had Миколаїв-Вантажний, a freight station)",
    ),
    "osm:n8220188327": ("Росинка", "fua:<city>"),
    "osm:n305264647": ("Севастополь", "fua:<city>"),
    "osm:n3693422903": ("Симферополь", "fua:<city>"),
    "osm:n11742271914": ("Херсон", "fua:<city>"),
    "osm:n4237097872": ("Шепетівка", "fua:<city>"),
    # --- TR ---
    "osm:n2726063373": ("Adana", "fua:<city>"),
    "osm:n1033799242": ("Denizli", "fua:<city>"),
    "osm:w88493479": ("Gaziantep Garı", "fua:<city>"),
    "osm:n1035592982": ("Isparta", "fua:<city>"),
    "osm:n5014804022": ("Konya", "fua:<city>"),
    "osm:n11082593543": ("Kırkikievler", "fua:<city>"),
    "osm:n2596542075": ("Mersin Garı", "fua:<city>"),
    "osm:n1550726413": ("Muş", "fua:<city>"),
    "osm:n125326513": ("Osmaniye", "fua:<city>"),
    "osm:n1023854236": ("Samsun", "tourism:<region>"),
    "osm:n13480008012": ("Zonguldak", "tourism:<region>"),
    "osm:n719596870": (
        "Полтава-Київська",
        "night_train_stop (absent from ONTD) — 19 trips",
    ),
    "osm:n1276842137": (
        "Кривий Ріг-Головний",
        "night_train_stop (replaces Кривий Ріг, a smaller station 3.8 km off)",
    ),
}

## Removed 2026-08-24 — duplicates of step 5

Step 6 was selected against a step 5 run that silently dropped 44 % of the
network, so several picks compensated for stops that are back, and a few
picked a metro/S-Bahn object sitting metres from the mainline station. The
guards below now block this class of entry; these were removed when the
guards were introduced (qualified counterpart in parentheses):

*Same OSM object, now qualified by step 5:* Remscheid-Lennep,
Rimini Torre Pedrera, Schönenwerd, Vignale-Riotorto, Централна гара Русе,
Централна гара София.

*Second OSM object for an already-qualified station:* Gesundbrunnen
(Berlin Gesundbrunnen), Spandau (Berlin-Spandau), Südkreuz (Berlin Südkreuz),
Euston (London Euston), King's Cross St Pancras (London King's Cross),
Gare du Midi (Bruxelles-Midi), Diamant (Antwerpen-Centraal),
Atocha-Cercanías (Madrid-Puerta de Atocha), Déli pályaudvar (Budapest-Déli),
Kelenföld vasútállomás (Budapest-Kelenföld), Lugano funicolare (Lugano),
Arlanda central (Arlanda norra), Вокзальна (Київ-Пасажирський).

*Not railway stations at all:* Vörösmarty utca (Budapest M1 metro; the FUA is
covered by Budapest-Nyugati), Hauptbahnhof Arnulf-Klett-Platz (Stuttgart
Stadtbahn; Stuttgart Hauptbahnhof is qualified 300 m away).

*Objects replaced with the mainline station:* Aarhus H (light-rail node →
`osm:n5026479524`), Athens (metro Σταθμός Λαρίσης → railway station
`osm:w609979867`).


## Combine, resolve and write

In [12]:
ADDITIONS = {}
for group in (
    ADDITIONS_GERMANY,
    ADDITIONS_FRANCE,
    ADDITIONS_IBERIA,
    ADDITIONS_ITALY,
    ADDITIONS_UNITED_KINGDOM,
    ADDITIONS_NORDICS,
    ADDITIONS_CENTRAL_EUROPE,
    ADDITIONS_BALTICS,
    ADDITIONS_SOUTH_EASTERN_EUROPE,
    ADDITIONS_EASTERN_EUROPE,
):
    overlap = ADDITIONS.keys() & group.keys()
    if overlap:
        raise ValueError(f"stop listed in two regions: {sorted(overlap)}")
    ADDITIONS.update(group)

print(f"manual additions: {len(ADDITIONS)}")

manual additions: 381


In [13]:
# Name and coordinates come from step 3b. Country prefers ONTD via step 4
# (curated national data); step 3b's own country column is only ~1% populated,
# so the handful of stops ONTD doesn't cover are listed explicitly below rather
# than pulled from the legacy step 6 file — a dozen values are not worth a file
# dependency, and here they are visible and reviewable.
MANUAL_COUNTRY = {
    "osm:r1821284": "IT",
    "osm:w529540422": "BG",
    "osm:w421067799": "BG",
    "osm:n14055863804": "BG",
    "osm:n13254223965": "BG",
    "osm:w28776478": "ES",
    "osm:w24930154": "ES",
    "osm:w86123754": "ES",
    "osm:w73575850": "FR",
    "osm:w112095036": "FR",
    "osm:w272187224": "RO",
    "osm:n202022357": "RO",
    "osm:r10274650": "SE",
    "osm:n1550726413": "TR",
    "osm:n247416268": "RO",
    "osm:w216482601": "BG",
    "osm:n4993961319": "DK",
    "osm:n60093107": "AT",
    "osm:n3129289404": "CZ",
    "osm:n24684084": "CZ",
    "osm:w609979867": "GR",
    "osm:r1821284": "IT",
}

osm = {}
with open(
    ensure_local("step3b_output_osm_stations_classified.csv"),
    encoding="utf-8-sig",
    newline="",
) as fh:
    for row in csv.DictReader(fh):
        osm[row["stop_id"]] = row

ontd_country = {}
ontd_of_osm = {}
with open(
    ensure_local("step4_MatchingONTDtoOSM.csv"), encoding="utf-8-sig", newline=""
) as fh:
    for row in csv.DictReader(fh):
        stop_id = (row.get("osm_stop_id") or "").strip()
        if not stop_id:
            continue
        country = (row.get("ontd_country") or "").strip().upper()
        if country:
            ontd_country.setdefault(stop_id, country)
        ontd_of_osm.setdefault(stop_id, row["ontd_id"])

unknown = sorted(set(ADDITIONS) - set(osm))
if unknown:
    raise KeyError(
        f"{len(unknown)} stop id(s) not in step 3b — typo, or the OSM extract was "
        f"refreshed and the object is gone: {unknown[:10]}"
    )

# --- guard 1: the picked object has to be a railway station -----------------
# Step 3b classifies but drops nothing, so metro, tram, bus and funicular
# objects sit in the same lookup — and metres from a mainline station they
# carry almost the same name. A ferry pier classifies as "other" and is
# legitimate, but only where the reason says the stop is there for the ferry.
wrong_mode = []
for stop_id, (label, reason) in ADDITIONS.items():
    mode = osm[stop_id]["station_mode"]
    ferry_pick = osm[stop_id]["mode_rule"] == "ferry_terminal" and reason.startswith(
        "ferry"
    )
    if mode == "urban_transit" or (mode == "other" and not ferry_pick):
        wrong_mode.append((label, stop_id, mode, osm[stop_id]["mode_rule"]))
if wrong_mode:
    for label, stop_id, mode, rule in wrong_mode:
        print(f"  {label[:40]:42} {stop_id:22} {mode} ({rule})")
    raise ValueError(
        f"{len(wrong_mode)} addition(s) point at a metro/tram/bus/funicular "
        "object, not the railway station — replace the id with the mainline "
        "station's (step 3b usually has it within 100 m of the picked object)"
    )

# --- guard 2: no addition may duplicate a step 5 stop -----------------------
# Step 10 dedups on OSM id alone, so a station qualifying through *different*
# OSM objects in the two layers would be written twice — the way the catalog
# once carried Gesundbrunnen beside Berlin Gesundbrunnen. Three tests, each an
# error: same OSM id, same ONTD station via the step 4 join, or within
# SAME_STATION_KM of a qualified stop (different objects for one station sit
# metres apart; distinct stations in one city do not).
SAME_STATION_KM = 0.3
SAME_AREA_KM = 15.0


def distance_km(lat1, lon1, lat2, lon2):
    # Equirectangular approximation — fine at station scale, cheap enough to
    # run every addition against every qualified stop without an index.
    mean_lat = math.radians((lat1 + lat2) / 2)
    dx = math.radians(lon2 - lon1) * math.cos(mean_lat)
    dy = math.radians(lat2 - lat1)
    return math.hypot(dx, dy) * 6371.0


step5 = []
with open(
    local_input("step5_JoinedNTStops.csv", "step5_JoinNTStopsWithOSM.ipynb"),
    encoding="utf-8-sig",
    newline="",
) as fh:
    for row in csv.DictReader(fh):
        if row["osm_lat"] and row["osm_lon"]:
            step5.append(
                {
                    "ontd_id": row["ontd_id"],
                    "stop_id": row["osm_stop_id"],
                    "name": row["osm_stop_name"] or row["ontd_name"],
                    "lat": float(row["osm_lat"]),
                    "lon": float(row["osm_lon"]),
                }
            )
qualified_ids = {s["stop_id"] for s in step5}
qualified_ontd = {s["ontd_id"] for s in step5}

duplicates, area_review = [], []
for stop_id, (label, reason) in ADDITIONS.items():
    lat = float(osm[stop_id]["stop_lat"])
    lon = float(osm[stop_id]["stop_lon"])
    d, nearest = min(
        ((distance_km(lat, lon, s["lat"], s["lon"]), s) for s in step5),
        key=lambda pair: pair[0],
    )
    if stop_id in qualified_ids:
        duplicates.append((label, stop_id, "same OSM id in step 5", nearest, d))
    elif ontd_of_osm.get(stop_id) in qualified_ontd:
        duplicates.append(
            (label, stop_id, "same ONTD station, different OSM object", nearest, d)
        )
    elif d <= SAME_STATION_KM:
        duplicates.append(
            (label, stop_id, f"{d * 1000:.0f} m from a qualified stop", nearest, d)
        )
    elif d <= SAME_AREA_KM and reason.startswith("fua"):
        area_review.append((label, stop_id, reason, nearest, d))

if duplicates:
    for label, stop_id, why, nearest, d in sorted(duplicates, key=lambda x: x[0]):
        print(f"  {label[:36]:38} {why:44} vs {nearest['name'][:32]} {stop_id}")
    raise ValueError(
        f"{len(duplicates)} addition(s) duplicate a step 5 stop — remove them, "
        "or replace the reason and id if a genuinely distinct station is meant"
    )

with open(OVERLAP_REVIEW_PATH, "w", encoding="utf-8-sig", newline="") as fh:
    writer = csv.DictWriter(
        fh,
        fieldnames=[
            "stop_id",
            "stop_name",
            "reason",
            "nearest_qualified",
            "nearest_stop_id",
            "distance_km",
        ],
    )
    writer.writeheader()
    for label, stop_id, reason, nearest, d in sorted(area_review, key=lambda x: -x[4]):
        writer.writerow(
            {
                "stop_id": stop_id,
                "stop_name": label,
                "reason": reason,
                "nearest_qualified": nearest["name"],
                "nearest_stop_id": nearest["stop_id"],
                "distance_km": f"{d:.1f}",
            }
        )
if area_review:
    print(
        f"{len(area_review)} fua additions have a qualified stop within "
        f"{SAME_AREA_KM:.0f} km — judgement calls, not errors; kept, and listed "
        f"in {OVERLAP_REVIEW_PATH.name}. If the second station is wanted, say "
        'so in the reason ("network:..."); an fua claim the data contradicts '
        "helps nobody."
    )

# --- resolve and write ------------------------------------------------------
rows = []
for stop_id, (label, reason) in ADDITIONS.items():
    station = osm[stop_id]
    rows.append(
        {
            "stop_id": stop_id,
            "stop_name": station["stop_name"].strip() or label,
            "country": (
                ontd_country.get(stop_id)
                or station["country"].strip().upper()
                or MANUAL_COUNTRY.get(stop_id, "")
            ),
            "stop_lat": station["stop_lat"],
            "stop_lon": station["stop_lon"],
            "reason": reason.strip(),
        }
    )
rows.sort(key=lambda r: (r["country"], r["stop_name"]))

no_country = [r for r in rows if not r["country"]]
if no_country:
    raise ValueError(
        f"{len(no_country)} addition(s) have no country — step 10 cannot derive "
        f"a timezone and would drop them: "
        f"{[(r['stop_id'], r['stop_name']) for r in no_country]}. "
        "Add them to MANUAL_COUNTRY above."
    )

with open(OUTPUT_PATH, "w", encoding="utf-8-sig", newline="") as fh:
    writer = csv.DictWriter(
        fh,
        fieldnames=[
            "stop_id",
            "stop_name",
            "country",
            "stop_lat",
            "stop_lon",
            "reason",
        ],
    )
    writer.writeheader()
    writer.writerows(rows)

print(f"wrote {len(rows)} rows to {OUTPUT_PATH.name}")
print("by country:", Counter(r["country"] for r in rows).most_common(8))

16 fua additions have a qualified stop within 15 km — judgement calls, not errors; kept, and listed in step6_overlap_review.csv. If the second station is wanted, say so in the reason ("network:..."); an fua claim the data contradicts helps nobody.
wrote 381 rows to step6_manual_additions.csv
by country: [('FR', 49), ('ES', 44), ('DE', 37), ('GB', 35), ('IT', 31), ('NL', 21), ('PL', 20), ('UA', 18)]


## Unfilled reasons

Everything listed here is a stop in the public catalog that cannot yet explain
why it is there — empty reasons and never-replaced `<...>` template
placeholders alike.


In [14]:
QUALIFIED_REASONS = {"fua", "tourism", "ferry", "border", "network", "night_train_stop"}
DETAILED_REASONS = {"fua", "tourism", "ferry"}


def reason_problem(reason: str) -> str | None:
    if not reason:
        return "empty"
    prefix, _, detail = reason.partition(":")
    prefix = prefix.split(" ")[0].split("\u2014")[0].strip()
    if prefix not in QUALIFIED_REASONS:
        return f"unknown criterion {prefix!r}"
    if prefix in DETAILED_REASONS:
        detail = detail.strip()
        if not detail:
            return f"'{prefix}' names no place"
        if "<" in detail or ">" in detail:
            return "template placeholder never filled in"
    return None


problems = [
    (row, problem) for row in rows if (problem := reason_problem(row["reason"]))
]
print(f"{len(problems)} of {len(rows)} additions cannot explain themselves\n")
for row, problem in problems:
    print(
        f"  {row['country']}  {row['stop_name'][:40]:42} {row['stop_id']:22} {problem}"
    )

356 of 381 additions cannot explain themselves

  AL  Durrës                                     osm:n13895194677       template placeholder never filled in
  AL  Terminali i Transportit Publik Tiranë      osm:n13895194676       template placeholder never filled in
  BA  Mostar                                     osm:n312280159         template placeholder never filled in
  BA  Sarajevo                                   osm:n942406094         template placeholder never filled in
  BA  Zenica                                     osm:n10212563520       template placeholder never filled in
  BE  Brugge                                     osm:n2929614444        template placeholder never filled in
  BE  Gent-Sint-Pieters                          osm:n1178257779        template placeholder never filled in
  BE  Kortrijk                                   osm:n21309047          template placeholder never filled in
  BE  La Louvière-Centre                         osm:n446059037         template